# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(meta.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"  RecordSet ID: {rs.id}, Name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field ID: {field.id}, Name: {field.name}, DataType: {field.data_type}")
    print()
if len(record_sets) > 0:
    print("Sample records from the first RecordSet:")
    rs0_id = record_sets[0].id
    for i, rec in enumerate(dataset.records(record_set=rs0_id)):
        print(rec)
        if i>=2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Select main record set and load all as dataframes by @id
df_dict = {}
record_set_ids = [rs.id for rs in meta.record_sets]
print('Record set ids:', record_set_ids)

for rec_id in record_set_ids:
    recs = list(dataset.records(record_set=rec_id))
    df = pd.DataFrame(recs)
    df_dict[rec_id] = df

# Choose the first one for demo/analysis
main_record_set_id = record_set_ids[0]
print(f"Columns in record set '{main_record_set_id}':")
print(df_dict[main_record_set_id].columns.tolist())
df_dict[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
# Identify a numeric field for analysis. Adjust this to your dataset:
numeric_columns = df_dict[main_record_set_id].select_dtypes(include=[np.number]).columns.tolist()
print('Numeric columns identified:', numeric_columns)
if len(numeric_columns) == 0:
    # Try finding a likely numeric field based on column names
    candidate = [c for c in df_dict[main_record_set_id].columns if 'count' in c.lower() or 'number' in c.lower() or 'abundance' in c.lower() or 'MW' in c or 'coverage' in c.lower()]
    numeric_field = candidate[0] if candidate else df_dict[main_record_set_id].columns[0]
else:
    numeric_field = numeric_columns[0]
print(f'Using numeric field: {numeric_field}')

df = df_dict[main_record_set_id].copy()
# Convert to numeric (pandas often loads as str from Croissant)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].mean()  # Example threshold
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by category if possible
categorical_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() < 20]
if len(categorical_candidates) > 0:
    group_field = categorical_candidates[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print("Grouped means:")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Simple visualization: histogram of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

if len(categorical_candidates) > 0:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded mass spectrometry protein data from extracellular vesicles.
- Reviewed record sets, fields, and performed EDA.
- Demonstrated filtering, normalization, and grouping by key attributes (when available).
- Visualizations showed the distribution and category relationships for a selected numeric field.

For deeper domain analysis, see the dataset documentation for suggested use cases and field meanings.